# 04.1.2 - Supervised Learning: Klasifikasi Akurasi Tinggi dengan SVM

## 1. Pengertian Support Vector Machine (SVM)
**Support Vector Machine (SVM)** adalah algoritma cerdas yang bertugas mencari "garis pemisah" terbaik di antara kelompok data yang berbeda. 

**Analogi Sederhana:**
Bayangkan kamu punya bola merah dan bola biru yang tersebar di atas meja. Tugas SVM adalah meletakkan sebuah penggaris (atau membangun tembok) tepat di tengah-tengah sedemikian rupa sehingga jarak tembok tersebut ke bola merah terdekat dan bola biru terdekat sama-sama maksimal. Tembok inilah yang nantinya dipakai untuk menebak bola baru yang dilempar ke meja: kalau jatuh di kiri tembok berarti merah, kalau di kanan berarti biru.

## 2. Mengapa Menggunakan SVM?
Pada tahap sebelumnya kita menggunakan *Decision Tree* untuk mengekstrak aturan bisnis yang mudah dibaca (*White-Box*). Lalu, kenapa sekarang kita pakai SVM yang sifatnya **Black-Box** (cara kerjanya rumit dan sulit dibaca manusia)?

Jawabannya adalah **Akurasi**. Model ini dibangun murni untuk menjadi "mesin penebak" kelompok pelanggan seakurat mungkin. Menurut referensi (Wang, 2025), SVM dengan *Kernel RBF* sangat direkomendasikan karena memberikan keseimbangan terbaik antara akurasi tebakan yang sangat tinggi dan kecepatan sistem. Model inilah yang nantinya akan kita pasang (integrasikan) ke dalam antarmuka Web UI.

## 3. Kapasitas Komputasi
Algoritma SVM memang dikenal sensitif terhadap ukuran data yang sangat besar (ratusan ribu baris). Namun, karena total dataset yang kita miliki saat ini relatif kecil (berada di kisaran 4.000-an baris), kita **tidak perlu melakukan pembatasan sampel**. Seluruh data latih akan digunakan agar model dapat menangkap karakteristik setiap cluster secara utuh dan menghasilkan performa yang maksimal.

In [ ]:
import pandas as pd
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# 1. BACA DATA (Gunakan Pemenang Clustering: Standard K-Means)
df = pd.read_csv('../data/Labeled/hasildata_kmeans-standard.csv')

# Pisahkan Fitur (Var1 - Var11) dan Target (Cluster)
fitur = [f'Var{i}' for i in range(1, 12)]
X = df[fitur]
y = df['Cluster']

# Splitting Data: 80% Train untuk belajar, 20% Test untuk ujian
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Menampilkan jumlah data untuk memastikan seluruh data digunakan
print(print(f"Jumlah data latih (Train): {X_train.shape[0]} baris"))
print(print(f"Jumlah data uji (Test): {X_test.shape[0]} baris"))

Jumlah data latih (Train): 3468 baris
None
Jumlah data uji (Test): 867 baris
None


## 4. Proses Scaling (Wajib untuk SVM)
Sebelum menyuruh SVM belajar, kita WAJIB melakukan *Scaling* (Penyetaraan Skala Data). Kenapa?

Data kebiasaan pelanggan punya rentang angka yang beda-beda. Misal, 'Jumlah Kunjungan' mungkin cuma puluhan (misal: 10 kali), tapi 'Total Belanja' bisa jutaan (misal: Rp 2.000.000). Kalau tidak disetarakan, model SVM akan mengira 'Total Belanja' jauh lebih penting daripada 'Jumlah Kunjungan' hanya karena angkanya lebih besar. 

Dengan `StandardScaler`, kita mengubah semua angka tersebut ke format seragam (skala standar) tanpa mengubah makna asli dari datanya.

In [5]:
# 2. SCALING DATA
scaler = StandardScaler()

# Model scaler belajar dari seluruh data latih (Train) sekaligus menyetarakan angkanya
X_train_scaled = scaler.fit_transform(X_train)

# Data ujian (Test) juga disetarakan menggunakan parameter dari data train di atas
X_test_scaled = scaler.transform(X_test)

## 5. Training Model SVM
Sekarang kita latih model menggunakan seluruh data train yang sudah disetarakan harganya. Kita menggunakan `kernel='rbf'` agar pembatas yang dibuat oleh model bisa melengkung secara fleksibel mengikuti bentuk sebaran cluster yang kompleks.

In [ ]:
# 3. TRAINING MODEL SVM
model_svm = SVC(kernel='rbf', random_state=42)
model_svm.fit(X_train_scaled, y_train)

# Evaluasi model menggunakan data ujian
prediksi_svm = model_svm.predict(X_test_scaled)
print("=== CLASSIFICATION REPORT: SVM (STANDARD K-MEANS) ===\n")
print(classification_report(y_test, prediksi_svm))

=== CLASSIFICATION REPORT: SVM (STANDARD K-MEANS) ===

              precision    recall  f1-score   support

           0       0.94      0.93      0.94       137
           1       0.83      0.91      0.87       211
           2       0.95      0.85      0.90       121
           3       0.98      0.93      0.95       166
           4       0.90      0.87      0.88       149
           5       0.92      0.99      0.95        83

    accuracy                           0.91       867
   macro avg       0.92      0.91      0.92       867
weighted avg       0.91      0.91      0.91       867



### Analisis Hasil
Rapor evaluasi (*Classification Report*) di atas menunjukkan tingkat kemampuan model dalam menebak kelas pelanggan baru secara akurat berdasarkan data uji. Model ini siap ditugaskan sebagai mesin pintar di balik layar Web UI.

## 6. Ekspor Model dan Scaler
Kita perlu menyimpan **dua file** penting ini agar bisa langsung dipanggil oleh sistem aplikasi tanpa perlu melatih ulang:
1. `scaler_svm_standard.pkl` (Alat penyetara angka input baru).
2. `model_svm_classification_standard.pkl` (Mesin penebaknya).

Alurnya nanti di Web UI: Ketika ada input data pelanggan baru, angka-angka tersebut wajib dilewatkan ke file `scaler` terlebih dahulu, baru kemudian hasilnya ditebak oleh file `model_svm`.

In [7]:
# 4. EXPORT SCALER & MODEL KE FOLDER 'models'
os.makedirs('../models', exist_ok=True)

# Simpan Scaler
joblib.dump(scaler, '../models/scaler_svm_standard.pkl')

# Simpan Model SVM
joblib.dump(model_svm, '../models/model_svm_classification_standard.pkl')

print("\n[SUCCESS] Scaler & Model SVM tanpa batasan sampel berhasil diekspor ke folder '../models/'!")


[SUCCESS] Scaler & Model SVM tanpa batasan sampel berhasil diekspor ke folder '../models/'!
